In [15]:
# Import packages and modules
import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow_decision_forests as tfdf

from sklearn.model_selection import train_test_split
from sklearn.svm import SVC

In [16]:
# Check the version of TensorFlow Decision Forests
print("Found TensorFlow Decision Forests v" + tfdf.__version__)

Found TensorFlow Decision Forests v1.9.1


In [17]:
# train/validation/test = 70/10/20
dataset = pd.read_csv('dataset/glrlm/kernel5-radius5/dataset.200p.csv')

train_data, temp_data = train_test_split(
    dataset, test_size=0.3, random_state=42
 )
validation_data, test_data = train_test_split(
    temp_data, test_size=2/3, random_state=42
 )

In [18]:
# Convert the dataset into a TensorFlow dataset.
train_ds = tfdf.keras.pd_dataframe_to_tf_dataset(
    train_data, label="label"
)         
val_ds = tfdf.keras.pd_dataframe_to_tf_dataset(
    validation_data, label="label"
)
test_ds = tfdf.keras.pd_dataframe_to_tf_dataset(
    test_data, label="label"
)

In [19]:
import keras

In [20]:
%%time

# Train an SVM model with class weight.
svm_model = SVC(kernel="rbf", probability=True, random_state=42, class_weight="balanced")
svm_model.fit(train_data.drop(columns=["label"]), train_data["label"])

CPU times: user 55min 15s, sys: 275 ms, total: 55min 15s
Wall time: 55min 15s


SVC(class_weight='balanced', probability=True, random_state=42)

In [21]:
# Evaluate the model with sklearn
from sklearn.metrics import accuracy_score

X_val = validation_data.drop(columns=["label"])
y_true = validation_data["label"].astype(int).to_numpy()
y_pred = svm_model.predict(X_val)

accuracy = accuracy_score(y_true, y_pred)
print(f"accuracy: {accuracy:.4f}")

accuracy: 0.8133


In [22]:
# Model Summary (sklearn style)
print(svm_model)
print(f"kernel: {svm_model.kernel}")
print(f"C: {svm_model.C}")
print(f"gamma: {svm_model.gamma}")

SVC(class_weight='balanced', probability=True, random_state=42)
kernel: rbf
C: 1.0
gamma: scale


In [23]:
# Model features
feature_names = train_data.drop(columns=["label"]).columns.tolist()
print("features:")
print(feature_names)

features:
['GrayLevelNonUniformity', 'GrayLevelNonUniformityNormalized', 'GrayLevelVariance', 'HighGrayLevelRunEmphasis', 'LongRunEmphasis', 'LongRunHighGrayLevelEmphasis', 'LongRunLowGrayLevelEmphasis', 'LowGrayLevelRunEmphasis', 'RunEntropy', 'RunLengthNonUniformity', 'RunLengthNonUniformityNormalized', 'RunPercentage', 'RunVariance', 'ShortRunEmphasis', 'ShortRunHighGrayLevelEmphasis', 'ShortRunLowGrayLevelEmphasis']


In [24]:
# Feature importance (not available for SVC with RBF kernel)
print("SVC with RBF kernel does not provide feature importances.")

SVC with RBF kernel does not provide feature importances.


In [25]:
# Model self evaluation (sklearn info)
print("number of support vectors:", svm_model.support_vectors_.shape[0])
print("support vectors per class:", svm_model.n_support_)

number of support vectors: 66498
support vectors per class: [62146  4352]


In [26]:
# Training logs (not available for sklearn SVC)
print("Training logs are not available for sklearn SVC.")

Training logs are not available for sklearn SVC.


Calculate the score of our hold-out validation dataset

In [27]:
X_val = validation_data.drop(columns=["label"])
y_true = validation_data["label"].astype(int).to_numpy()
pos_probs = svm_model.predict_proba(X_val)[:, 1]

from sklearn.metrics import roc_auc_score
ROC_AUC = roc_auc_score(y_true, pos_probs)
print("The ROC AUC score is %.5f" % ROC_AUC )

The ROC AUC score is 0.91082


In [28]:
# Compute binary classification metrics with sklearn
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report,
    matthews_corrcoef,
    log_loss,
    brier_score_loss,
 )

# Get predictions (probabilities or class labels)
X_val = test_data.drop(columns=["label"])
y_true = test_data["label"].astype(int).to_numpy()
pos_probs = svm_model.predict_proba(X_val)[:, 1]
y_pred = (pos_probs >= 0.5).astype(int)

# Core metrics
metrics = {}
metrics["accuracy"] = accuracy_score(y_true, y_pred)
metrics["precision"] = precision_score(y_true, y_pred, zero_division=0)
metrics["recall"] = recall_score(y_true, y_pred, zero_division=0)
metrics["f1"] = f1_score(y_true, y_pred, zero_division=0)
metrics["mcc"] = matthews_corrcoef(y_true, y_pred)

# Probabilistic metrics
metrics["roc_auc"] = roc_auc_score(y_true, pos_probs)
metrics["pr_auc"] = average_precision_score(y_true, pos_probs)
metrics["log_loss"] = log_loss(y_true, pos_probs, labels=[0,1])
metrics["brier_score"] = brier_score_loss(y_true, pos_probs)

# Confusion matrix and detailed report
cm = confusion_matrix(y_true, y_pred, labels=[0,1])
report = classification_report(y_true, y_pred, digits=4)

print("Sklearn binary metrics:")
for k, v in metrics.items():
    print(f"{k}: {v:.4f}")
print("\nConfusion matrix:\n", cm)
print("\nClassification report:\n", report)

Sklearn binary metrics:
accuracy: 0.9410
precision: 0.6309
recall: 0.1814
f1: 0.2818
mcc: 0.3173
roc_auc: 0.9063
pr_auc: 0.4497
log_loss: 0.1549
brier_score: 0.0445

Confusion matrix:
 [[41610   303]
 [ 2337   518]]

Classification report:
               precision    recall  f1-score   support

           0     0.9468    0.9928    0.9693     41913
           1     0.6309    0.1814    0.2818      2855

    accuracy                         0.9410     44768
   macro avg     0.7889    0.5871    0.6255     44768
weighted avg     0.9267    0.9410    0.9254     44768

